# Sonata + Cosine Masking — Pure pip, Pinned VersionsNo conda, no JIT compilation, no restart required.All versions pinned to match the official environment.yml.

In [ ]:
# CELL 1 — Install Everything (5 min, NO restart needed)import os, subprocess# 1. Check Colab environment!nvcc --version 2>/dev/null | grep release!python --version# 2. PyTorch 2.5.0 + CUDA 12.4 (exact match to environment.yml)!pip install -q torch==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cu124# 3. spconv — prebuilt wheel, no compilation needed!pip install -q spconv-cu124# 4. System deps for pointgroup_ops!apt-get install -y -qq libsparsehash-dev 2>&1 | tail -1# 5. Python deps!pip install -q ninja h5py addict pyyaml tensorboard timm peft tqdm!pip install -q torch-cluster torch-scatter -f https://data.pyg.org/whl/torch-2.5.0+cu124.html# 6. Clone + compile CUDA ops!rm -rf /content/voxel_group_classifier!git clone --depth 1 https://github.com/yuhang-wang-xjtu/voxel_group_classifier.git /content/voxel_group_classifier%cd /content/voxel_group_classifier# Compile custom CUDA extensions (one-time, ~2 min)for lib in pointops pointops2 pointgroup_ops; do    echo "Building $lib..."    !cd libs/$lib && MAX_JOBS=2 pip install . --quiet --no-build-isolation 2>&1 | tail -1done%cd /content/voxel_group_classifier# 7. Quick sanity checkimport torch, spconvprint(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | spconv {spconv.__version__}")x = torch.rand(1, 3, 64, 64, 64).cuda()y = spconv.SubMConv3d(3, 3, 3, bias=False).cuda()(x)print(f"spconv forward: {y.shape}  OK")!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# CELL 2 — Download S3DIS HDF5 + Convert to Pointcept format (~3 min)import numpy as np, h5py, os, glob, urllib.request, zipfileHDF5_URL = "https://huggingface.co/datasets/cminst/S3DIS/resolve/main/indoor3d_sem_seg_hdf5_data.zip"HDF5_DIR = "./indoor3d_sem_seg_hdf5_data"OUT_ROOT = "./data/s3dis"if not os.path.isdir(HDF5_DIR):    print("Downloading S3DIS (~1.7 GB)...")    urllib.request.urlretrieve(HDF5_URL, "/tmp/s3dis.zip")    with zipfile.ZipFile("/tmp/s3dis.zip") as zf:        zf.extractall(HDF5_DIR)h5_files = sorted(glob.glob(f"{HDF5_DIR}/*/*.h5"))print(f"Converting {len(h5_files)} HDF5 files...")for h5_path in h5_files:    fname = os.path.basename(h5_path).replace(".h5", "")    area_idx = int(fname.split("_")[-1])    with h5py.File(h5_path, "r") as f:        data, labels = f["data"][:], f["label"][:]    room_dir = os.path.join(OUT_ROOT, f"Area_{area_idx}", fname)    os.makedirs(room_dir, exist_ok=True)    np.save(os.path.join(room_dir, "coord.npy"), data[..., :3].reshape(-1, 3).astype(np.float32))    np.save(os.path.join(room_dir, "color.npy"), data[..., 3:6].reshape(-1, 3).astype(np.uint8))    np.save(os.path.join(room_dir, "segment.npy"), labels.reshape(-1, 1).astype(np.int16))    np.save(os.path.join(room_dir, "instance.npy"), np.zeros_like(labels.reshape(-1, 1), dtype=np.int16))print(f"Done. {OUT_ROOT}/ ready.")

In [ ]:
# CELL 3 — Train (cosine masking, 10 epoch smoke test)MASKING_MODE = "cosine"EPOCHS = 10CONFIG = ("configs/sonata/pretrain-sonata-v1m2-cosine-smoketest.py"         if MASKING_MODE == "cosine"         else "configs/sonata/pretrain-sonata-v1m2-vgc-smoketest.py")SAVE = f"exp/{MASKING_MODE}_{EPOCHS}ep"print(f"Config: {CONFIG}  |  Save: {SAVE}")!python tools/train.py --config-file {CONFIG} --options save_path={SAVE} epoch={EPOCHS} data.train.datasets.0.data_root=data/s3dis enable_wandb=False# Monitor in separate cell with: %load_ext tensorboard; %tensorboard --logdir exp